In [1]:
import pandas as pd
import numpy as np
import re
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Load data - sử dụng file merged từ 3 nguồn
df = pd.read_csv('../data/all_recipes_final.csv')
print(f"Dataset shape: {df.shape}")
print(f"\n📊 Phân bố theo nguồn:")
print(df['source'].value_counts())
df.head()

Dataset shape: (3524, 9)

📊 Phân bố theo nguồn:
source
monngonmoingay    2386
vnexpress          886
dienmayxanh        252
Name: count, dtype: int64


,title,description,ingredients,step,calories,cook_time,type_of_food,url,source
0,Cách muối dưa hành truyền thống,Dưa hành muối là món ăn truyền thống ngày Tết ...,"['1 kg hành củ tươi', 'Tro bếp hoặc nước vo gọ...",['Bước 1: Chọn hành củ: Nên chọn hành củ ta bá...,459 kcal,45 phút,Món Tết,https://vnexpress.net/doi-song-cooking-cach-mu...,vnexpress
1,Su hào xào mực - món cổ Tết Bát Tràng,Đĩa xào khô ráo với su hào giòn ngọt quyện với...,"['2 củ su hào non', '1 con mực khô', '1/2 củ c...",['Bước 1: Chọn và sơ chế mực: Người dân làng g...,1.162 kcal,50 phút,Món Tết,https://vnexpress.net/doi-song-cooking-su-hao-...,vnexpress
2,Canh măng ngày Tết cổ truyền Hà Nội,"Măng ngấu vị, giòn ngon, móng giò hầm vừa độ s...","['800 gr măng khô', '2 móng giò lợn', 'Nước dù...","['Bước 1: Chọn măng khô: Theo lối cũ, người nộ...",4.930 kcal,100 phút,Món Tết,https://vnexpress.net/doi-song-cooking-canh-ma...,vnexpress
3,Giả hạnh nhân - món ngon Tết xưa Hà Nội,Đây là món ăn cổ truyền thường thấy trong cỗ T...,"['2 bộ lòng mề gà', '100 gr lạc', '50 gr hạt đ...",['Bước 1: Chọn và sơ chế lạc: Chọn lạc khô chắ...,1.112 kcal,60 phút,Món Tết,https://vnexpress.net/doi-song-cooking-gia-han...,vnexpress
4,Chả bì ớt xiêm xanh,"Chả bì bóng đẹp, gói đều tay. Khi ăn vị ngọt m...","['500 gr giò sống', '300 gr bì lợn', '20 - 30 ...","['Bước 1: Chọn và sơ chế bì lợn, chuẩn bị giò ...",2.512 kcal,60 phút,Món Tết,https://vnexpress.net/doi-song-cooking-cha-bi-...,vnexpress


In [3]:
# Xem thông tin dữ liệu
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3524 entries, 0 to 3523
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         3524 non-null   object
 1   description   3448 non-null   object
 2   ingredients   3524 non-null   object
 3   step          3524 non-null   object
 4   calories      437 non-null    object
 5   cook_time     3080 non-null   object
 6   type_of_food  3524 non-null   object
 7   url           3524 non-null   object
 8   source        3524 non-null   object
dtypes: object(9)
memory usage: 247.9+ KB


## 1. Data Preprocessing

In [4]:
# Tạo bản copy để xử lý
data = df.copy()

# Xử lý missing values
data['title'] = data['title'].fillna('')
data['description'] = data['description'].fillna('')
data['step'] = data['step'].fillna('[]')
data['ingredients'] = data['ingredients'].fillna('[]')
data['type_of_food'] = data['type_of_food'].fillna('Unknown')

print("Missing values after handling:")
print(data[['title', 'description', 'step', 'ingredients', 'type_of_food', 'calories', 'cook_time']].isnull().sum())

Missing values after handling:
title              0
description        0
step               0
ingredients        0
type_of_food       0
calories        3087
cook_time        444
dtype: int64


In [5]:
def parse_list_string(s):
    """Parse string representation of list to actual list"""
    if pd.isna(s) or s == '[]':
        return []
    try:
        return ast.literal_eval(s)
    except:
        return []

def clean_text(text):
    """Clean and normalize text"""
    if pd.isna(text):
        return ''
    text = str(text).lower()
    # Remove special characters but keep Vietnamese characters
    text = re.sub(r'[^\w\s\u00C0-\u1EF9]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def parse_cook_time(time_str):
    """Parse cook time string to minutes"""
    if pd.isna(time_str):
        return np.nan
    time_str = str(time_str).lower()
    minutes = 0
    
    # Find hours
    hour_match = re.search(r'(\d+)\s*(?:giờ|h|hour)', time_str)
    if hour_match:
        minutes += int(hour_match.group(1)) * 60
    
    # Find minutes
    min_match = re.search(r'(\d+)\s*(?:phút|p|min|minute)', time_str)
    if min_match:
        minutes += int(min_match.group(1))
    
    # If only number found
    if minutes == 0:
        num_match = re.search(r'(\d+)', time_str)
        if num_match:
            minutes = int(num_match.group(1))
    
    return minutes if minutes > 0 else np.nan

def parse_calories(cal_str):
    """Parse calories string to numeric"""
    if pd.isna(cal_str):
        return np.nan
    cal_str = str(cal_str).replace('.', '').replace(',', '')
    match = re.search(r'(\d+)', cal_str)
    if match:
        return float(match.group(1))
    return np.nan

In [6]:
# Apply preprocessing
# Parse ingredients and steps
data['ingredients_list'] = data['ingredients'].apply(parse_list_string)
data['step_list'] = data['step'].apply(parse_list_string)

# Clean text fields
data['title_clean'] = data['title'].apply(clean_text)
data['description_clean'] = data['description'].apply(clean_text)
data['step_clean'] = data['step_list'].apply(lambda x: ' '.join([clean_text(s) for s in x]))

# Parse numeric fields
data['cook_time_minutes'] = data['cook_time'].apply(parse_cook_time)
data['calories_numeric'] = data['calories'].apply(parse_calories)

# Clean ingredients for Jaccard
data['ingredients_clean'] = data['ingredients_list'].apply(
    lambda x: set([clean_text(ing) for ing in x if ing])
)

print("Preprocessing completed!")
data[['title', 'title_clean', 'cook_time', 'cook_time_minutes', 'calories', 'calories_numeric']].head()

Preprocessing completed!


,title,title_clean,cook_time,cook_time_minutes,calories,calories_numeric
0,Cách muối dưa hành truyền thống,cách muối dưa hành truyền thống,45 phút,45.0,459 kcal,459.0
1,Su hào xào mực - món cổ Tết Bát Tràng,su hào xào mực món cổ tết bát tràng,50 phút,50.0,1.162 kcal,1162.0
2,Canh măng ngày Tết cổ truyền Hà Nội,canh măng ngày tết cổ truyền hà nội,100 phút,100.0,4.930 kcal,4930.0
3,Giả hạnh nhân - món ngon Tết xưa Hà Nội,giả hạnh nhân món ngon tết xưa hà nội,60 phút,60.0,1.112 kcal,1112.0
4,Chả bì ớt xiêm xanh,chả bì ớt xiêm xanh,60 phút,60.0,2.512 kcal,2512.0


In [7]:
# Kiểm tra kết quả preprocessing
print("Sample ingredients_clean:")
for i, ing in enumerate(data['ingredients_clean'].head(3)):
    print(f"\nRecipe {i+1}: {data['title'].iloc[i]}")
    print(f"Ingredients: {ing}")

Sample ingredients_clean:

Recipe 1: Cách muối dưa hành truyền thống
Ingredients: {'muối hạt đường', 'lọ sạch', 'tro bếp hoặc nước vo gọa', 'cà rốt trang trí tùy chọn', '1 kg hành củ tươi'}

Recipe 2: Su hào xào mực - món cổ Tết Bát Tràng
Ingredients: {'mỡ lợn hoặc dầu ăn', 'rau mùi trang trí', '1 con mực khô', 'gia vị mắm muối đường hạt tiêu rượu trắng gừng', '1 2 củ cà rốt', '2 củ su hào non'}

Recipe 3: Canh măng ngày Tết cổ truyền Hà Nội
Ingredients: {'nước vo gạo ngâm măng', 'gia vị nước mắm truyền thống muối', 'hành khô hành củ', '800 gr măng khô', '2 móng giò lợn', 'nước dùng gà hoặc ninh xương lợn'}


## 2. TF-IDF Based Recommendation

Sử dụng TF-IDF vectorizer trên text content (title + description + steps) và tính cosine similarity

In [8]:
# Combine text features
data['combined_text'] = data['title_clean'] + ' ' + data['description_clean'] + ' ' + data['step_clean']

# TF-IDF Vectorization
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),  # unigrams and bigrams
    min_df=1,
    max_df=0.95,
    stop_words=None  # Keep Vietnamese words
)

tfidf_matrix = tfidf_vectorizer.fit_transform(data['combined_text'])
print(f"TF-IDF Matrix shape: {tfidf_matrix.shape}")

TF-IDF Matrix shape: (3524, 5000)


In [9]:
# Compute TF-IDF cosine similarity matrix
tfidf_similarity = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f"TF-IDF Similarity Matrix shape: {tfidf_similarity.shape}")

TF-IDF Similarity Matrix shape: (3524, 3524)


In [10]:
def get_tfidf_recommendations(recipe_idx, similarity_matrix, df, top_n=5):
    """
    Get top N similar recipes based on TF-IDF similarity
    
    Parameters:
    - recipe_idx: Index of the recipe to find recommendations for
    - similarity_matrix: Precomputed similarity matrix
    - df: DataFrame containing recipe data
    - top_n: Number of recommendations to return
    
    Returns:
    - DataFrame with recommended recipes and similarity scores
    """
    # Get similarity scores for the given recipe
    sim_scores = list(enumerate(similarity_matrix[recipe_idx]))
    
    # Sort by similarity (descending)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Get top N (excluding the recipe itself)
    sim_scores = sim_scores[1:top_n+1]
    
    # Get recipe indices
    recipe_indices = [i[0] for i in sim_scores]
    scores = [i[1] for i in sim_scores]
    
    # Create result DataFrame
    result = df.iloc[recipe_indices][['title', 'type_of_food', 'calories', 'cook_time']].copy()
    result['similarity_score'] = scores
    
    return result

def recommend_by_title_tfidf(title, df, similarity_matrix, top_n=5):
    """
    Get recommendations by recipe title
    """
    # Find recipe index by title
    matches = df[df['title'].str.contains(title, case=False, na=False)]
    if len(matches) == 0:
        print(f"No recipe found with title containing: {title}")
        return None
    
    recipe_idx = matches.index[0]
    print(f"\n🍽️ Input Recipe: {df.loc[recipe_idx, 'title']}")
    print(f"   Type: {df.loc[recipe_idx, 'type_of_food']}")
    print(f"   Calories: {df.loc[recipe_idx, 'calories']}")
    print("\n📋 TF-IDF Recommendations:")
    
    return get_tfidf_recommendations(recipe_idx, similarity_matrix, df, top_n)

In [11]:
# Test TF-IDF Recommendation
print("METHOD 1: TF-IDF BASED RECOMMENDATION")

# Test với một món ăn
test_title = "Canh măng"
recommendations = recommend_by_title_tfidf(test_title, data, tfidf_similarity, top_n=5)
if recommendations is not None:
    display(recommendations)

METHOD 1: TF-IDF BASED RECOMMENDATION

🍽️ Input Recipe: Canh măng ngày Tết cổ truyền Hà Nội
   Type: Món Tết
   Calories: 4.930 kcal

📋 TF-IDF Recommendations:


,title,type_of_food,calories,cook_time,similarity_score
65,Canh móng giò hầm măng khô kiểu Bắc,Món Tết,2.390 kcal,90 phút,0.581095
155,Cách làm canh măng mực - đặc sản truyền thống ...,Món ngon hàng ngày,1.190 kcal,120 phút,0.484022
28,Cách làm thịt kho măng khô- món ngon Tết miền ...,Món Tết,3.043 kcal,75 phút,0.442955
330,Cách làm miến vịt măng khô đơn giản tại nhà,Món ngon hàng ngày,3.520 kcal,90 phút,0.442795
3340,"2 cách làm gà kho măng thơm ngon, bắt vị cho b...",Món Kho,NaN,30 phút + 40 phút + 15 phút + 15 phút,0.419960


## 3. Ingredient-Based Recommendation (Jaccard Similarity)

Sử dụng Jaccard Similarity để so sánh sự tương đồng giữa các thành phần nguyên liệu

In [12]:
def jaccard_similarity(set1, set2):
    """
    Calculate Jaccard similarity between two sets
    J(A,B) = |A ∩ B| / |A ∪ B|
    """
    if len(set1) == 0 and len(set2) == 0:
        return 0.0
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return intersection / union if union > 0 else 0.0

def compute_jaccard_similarity_matrix(ingredients_list):
    """
    Compute Jaccard similarity matrix for all recipes
    """
    n = len(ingredients_list)
    similarity_matrix = np.zeros((n, n))
    
    for i in range(n):
        for j in range(i, n):
            sim = jaccard_similarity(ingredients_list[i], ingredients_list[j])
            similarity_matrix[i][j] = sim
            similarity_matrix[j][i] = sim
    
    return similarity_matrix

In [13]:
# Compute Jaccard similarity matrix
print("Computing Jaccard similarity matrix...")
ingredients_sets = data['ingredients_clean'].tolist()
jaccard_similarity_matrix = compute_jaccard_similarity_matrix(ingredients_sets)
print(f"Jaccard Similarity Matrix shape: {jaccard_similarity_matrix.shape}")

Computing Jaccard similarity matrix...
Jaccard Similarity Matrix shape: (3524, 3524)
Jaccard Similarity Matrix shape: (3524, 3524)


In [14]:
def get_ingredient_recommendations(recipe_idx, similarity_matrix, df, top_n=5):
    """
    Get top N similar recipes based on ingredient (Jaccard) similarity
    """
    sim_scores = list(enumerate(similarity_matrix[recipe_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    
    recipe_indices = [i[0] for i in sim_scores]
    scores = [i[1] for i in sim_scores]
    
    result = df.iloc[recipe_indices][['title', 'type_of_food', 'calories', 'cook_time']].copy()
    result['jaccard_score'] = scores
    
    return result

def recommend_by_title_jaccard(title, df, similarity_matrix, top_n=5):
    """
    Get ingredient-based recommendations by recipe title
    """
    matches = df[df['title'].str.contains(title, case=False, na=False)]
    if len(matches) == 0:
        print(f"No recipe found with title containing: {title}")
        return None
    
    recipe_idx = matches.index[0]
    print(f"\n🍽️ Input Recipe: {df.loc[recipe_idx, 'title']}")
    print(f"   Ingredients: {list(df.loc[recipe_idx, 'ingredients_clean'])[:5]}...")
    print("\n📋 Ingredient-Based (Jaccard) Recommendations:")
    
    return get_ingredient_recommendations(recipe_idx, similarity_matrix, df, top_n)

In [15]:
# Test Ingredient-Based Recommendation
print("METHOD 2: INGREDIENT-BASED RECOMMENDATION (JACCARD)")

test_title = "Canh măng"
recommendations = recommend_by_title_jaccard(test_title, data, jaccard_similarity_matrix, top_n=5)
if recommendations is not None:
    display(recommendations)

METHOD 2: INGREDIENT-BASED RECOMMENDATION (JACCARD)

🍽️ Input Recipe: Canh măng ngày Tết cổ truyền Hà Nội
   Ingredients: ['nước vo gạo ngâm măng', 'gia vị nước mắm truyền thống muối', 'hành khô hành củ', '800 gr măng khô', '2 móng giò lợn']...

📋 Ingredient-Based (Jaccard) Recommendations:


,title,type_of_food,calories,cook_time,jaccard_score
65,Canh móng giò hầm măng khô kiểu Bắc,Món Tết,2.390 kcal,90 phút,0.090909
0,Cách muối dưa hành truyền thống,Món Tết,459 kcal,45 phút,0.000000
1,Su hào xào mực - món cổ Tết Bát Tràng,Món Tết,1.162 kcal,50 phút,0.000000
3,Giả hạnh nhân - món ngon Tết xưa Hà Nội,Món Tết,1.112 kcal,60 phút,0.000000
4,Chả bì ớt xiêm xanh,Món Tết,2.512 kcal,60 phút,0.000000


## 4. Metadata Similarity

Tính toán similarity dựa trên metadata: calories, cook_time, type_of_food

In [16]:
def compute_metadata_similarity_matrix(df):
    """
    Compute similarity matrix based on metadata features:
    - calories (normalized)
    - cook_time (normalized)
    - type_of_food (one-hot encoded)
    """
    n = len(df)
    
    # Normalize numeric features
    scaler = MinMaxScaler()
    
    # Handle missing values
    calories = df['calories_numeric'].fillna(df['calories_numeric'].median()).values.reshape(-1, 1)
    cook_time = df['cook_time_minutes'].fillna(df['cook_time_minutes'].median()).values.reshape(-1, 1)
    
    calories_normalized = scaler.fit_transform(calories)
    cook_time_normalized = scaler.fit_transform(cook_time)
    
    # One-hot encode type_of_food
    type_dummies = pd.get_dummies(df['type_of_food'], prefix='type')
    
    # Combine all features
    metadata_features = np.hstack([
        calories_normalized,
        cook_time_normalized,
        type_dummies.values
    ])
    
    print(f"Metadata features shape: {metadata_features.shape}")
    
    # Compute cosine similarity
    similarity_matrix = cosine_similarity(metadata_features, metadata_features)
    
    return similarity_matrix, metadata_features

In [17]:
# Compute metadata similarity
print("Computing Metadata similarity matrix...")
metadata_similarity_matrix, metadata_features = compute_metadata_similarity_matrix(data)
print(f"Metadata Similarity Matrix shape: {metadata_similarity_matrix.shape}")

Computing Metadata similarity matrix...
Metadata features shape: (3524, 28)
Metadata Similarity Matrix shape: (3524, 3524)


In [18]:
def get_metadata_recommendations(recipe_idx, similarity_matrix, df, top_n=5):
    """
    Get top N similar recipes based on metadata similarity
    """
    sim_scores = list(enumerate(similarity_matrix[recipe_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    
    recipe_indices = [i[0] for i in sim_scores]
    scores = [i[1] for i in sim_scores]
    
    result = df.iloc[recipe_indices][['title', 'type_of_food', 'calories', 'cook_time']].copy()
    result['metadata_score'] = scores
    
    return result

def recommend_by_title_metadata(title, df, similarity_matrix, top_n=5):
    """
    Get metadata-based recommendations by recipe title
    """
    matches = df[df['title'].str.contains(title, case=False, na=False)]
    if len(matches) == 0:
        print(f"No recipe found with title containing: {title}")
        return None
    
    recipe_idx = matches.index[0]
    print(f"\n🍽️ Input Recipe: {df.loc[recipe_idx, 'title']}")
    print(f"   Type: {df.loc[recipe_idx, 'type_of_food']}")
    print(f"   Calories: {df.loc[recipe_idx, 'calories']}")
    print(f"   Cook time: {df.loc[recipe_idx, 'cook_time']}")
    print("\n📋 Metadata-Based Recommendations:")
    
    return get_metadata_recommendations(recipe_idx, similarity_matrix, df, top_n)

In [19]:
# Test Metadata-Based Recommendation
print("METADATA-BASED RECOMMENDATION")

test_title = "Canh măng"
recommendations = recommend_by_title_metadata(test_title, data, metadata_similarity_matrix, top_n=5)
if recommendations is not None:
    display(recommendations)

METADATA-BASED RECOMMENDATION

🍽️ Input Recipe: Canh măng ngày Tết cổ truyền Hà Nội
   Type: Món Tết
   Calories: 4.930 kcal
   Cook time: 100 phút

📋 Metadata-Based Recommendations:


,title,type_of_food,calories,cook_time,metadata_score
51,Cách làm mứt dừa truyền thống đơn giản,Món Tết,4.420 kcal,100 phút,0.999737
62,Mứt me gọi Tết miền Tây,Món Tết,4.000 kcal,120 phút,0.998730
77,Sườn tảng nướng cam cho tiệc Giáng sinh,Món Tết,3.738 kcal,90 phút,0.998469
73,Cách làm mứt vỏ cam đón Tết,Món Tết,3.400 kcal,90 phút,0.997515
56,"Hướng dẫn cách nấu xôi gấc ngon, dẻo, đẹp",Món Tết,3.849 kcal,60 phút,0.997398


## 5. Hybrid Approach (Weighted Combination)

Kết hợp 3 phương pháp với trọng số:
- Text Similarity (TF-IDF): **0.4**
- Ingredient Similarity (Jaccard): **0.4**
- Metadata Similarity: **0.2**

In [20]:
def compute_hybrid_similarity(tfidf_sim, jaccard_sim, metadata_sim, 
                              w_tfidf=0.4, w_jaccard=0.4, w_metadata=0.2):
    """
    Compute hybrid similarity matrix as weighted combination
    
    Parameters:
    - tfidf_sim: TF-IDF similarity matrix
    - jaccard_sim: Jaccard similarity matrix
    - metadata_sim: Metadata similarity matrix
    - w_tfidf: Weight for TF-IDF (default 0.4)
    - w_jaccard: Weight for Jaccard (default 0.4)
    - w_metadata: Weight for metadata (default 0.2)
    
    Returns:
    - Hybrid similarity matrix
    """
    # Ensure weights sum to 1
    total_weight = w_tfidf + w_jaccard + w_metadata
    w_tfidf /= total_weight
    w_jaccard /= total_weight
    w_metadata /= total_weight
    
    print(f"Weights: TF-IDF={w_tfidf:.2f}, Jaccard={w_jaccard:.2f}, Metadata={w_metadata:.2f}")
    
    hybrid_similarity = (w_tfidf * tfidf_sim + 
                        w_jaccard * jaccard_sim + 
                        w_metadata * metadata_sim)
    
    return hybrid_similarity

In [21]:
# Compute hybrid similarity matrix
print("Computing Hybrid similarity matrix...")
hybrid_similarity_matrix = compute_hybrid_similarity(
    tfidf_similarity, 
    jaccard_similarity_matrix, 
    metadata_similarity_matrix,
    w_tfidf=0.4,
    w_jaccard=0.4,
    w_metadata=0.2
)
print(f"Hybrid Similarity Matrix shape: {hybrid_similarity_matrix.shape}")

Computing Hybrid similarity matrix...
Weights: TF-IDF=0.40, Jaccard=0.40, Metadata=0.20
Hybrid Similarity Matrix shape: (3524, 3524)
Hybrid Similarity Matrix shape: (3524, 3524)


In [22]:
def get_hybrid_recommendations(recipe_idx, hybrid_sim, tfidf_sim, jaccard_sim, metadata_sim, df, top_n=5):
    """
    Get top N similar recipes based on hybrid similarity with detailed scores
    """
    sim_scores = list(enumerate(hybrid_sim[recipe_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    
    recipe_indices = [i[0] for i in sim_scores]
    hybrid_scores = [i[1] for i in sim_scores]
    
    result = df.iloc[recipe_indices][['title', 'type_of_food', 'calories', 'cook_time']].copy()
    result['hybrid_score'] = hybrid_scores
    result['tfidf_score'] = [tfidf_sim[recipe_idx][i] for i in recipe_indices]
    result['jaccard_score'] = [jaccard_sim[recipe_idx][i] for i in recipe_indices]
    result['metadata_score'] = [metadata_sim[recipe_idx][i] for i in recipe_indices]
    
    return result

def recommend_by_title_hybrid(title, df, hybrid_sim, tfidf_sim, jaccard_sim, metadata_sim, top_n=5):
    """
    Get hybrid recommendations by recipe title
    """
    matches = df[df['title'].str.contains(title, case=False, na=False)]
    if len(matches) == 0:
        print(f"No recipe found with title containing: {title}")
        return None
    
    recipe_idx = matches.index[0]
    print(f"\n🍽️ Input Recipe: {df.loc[recipe_idx, 'title']}")
    print(f"   Type: {df.loc[recipe_idx, 'type_of_food']}")
    print(f"   Calories: {df.loc[recipe_idx, 'calories']}")
    print(f"   Cook time: {df.loc[recipe_idx, 'cook_time']}")
    print(f"   Ingredients: {list(df.loc[recipe_idx, 'ingredients_clean'])[:3]}...")
    print("\n📋 Hybrid Recommendations (TF-IDF: 0.4 + Jaccard: 0.4 + Metadata: 0.2):")
    
    return get_hybrid_recommendations(recipe_idx, hybrid_sim, tfidf_sim, jaccard_sim, metadata_sim, df, top_n)

In [23]:
# Test Hybrid Recommendation
print("METHOD 3: HYBRID APPROACH (WEIGHTED COMBINATION)")

test_title = "Canh măng"
recommendations = recommend_by_title_hybrid(
    test_title, data, 
    hybrid_similarity_matrix, 
    tfidf_similarity, 
    jaccard_similarity_matrix, 
    metadata_similarity_matrix, 
    top_n=5
)
if recommendations is not None:
    display(recommendations)

METHOD 3: HYBRID APPROACH (WEIGHTED COMBINATION)

🍽️ Input Recipe: Canh măng ngày Tết cổ truyền Hà Nội
   Type: Món Tết
   Calories: 4.930 kcal
   Cook time: 100 phút
   Ingredients: ['nước vo gạo ngâm măng', 'gia vị nước mắm truyền thống muối', 'hành khô hành củ']...

📋 Hybrid Recommendations (TF-IDF: 0.4 + Jaccard: 0.4 + Metadata: 0.2):


,title,type_of_food,calories,cook_time,hybrid_score,tfidf_score,jaccard_score,metadata_score
65,Canh móng giò hầm măng khô kiểu Bắc,Món Tết,2.390 kcal,90 phút,0.467437,0.581095,0.090909,0.993178
28,Cách làm thịt kho măng khô- món ngon Tết miền ...,Món Tết,3.043 kcal,75 phút,0.376339,0.442955,0.000000,0.995786
13,Cách làm canh bóng thả Hà Nội nấu theo lối xưa,Món Tết,1.157 kcal,85 phút,0.305155,0.270534,0.000000,0.984709
16,Cách làm giò xào truyền thống kiểu Bắc,Món Tết,3.082 kcal,75 phút,0.297050,0.244655,0.000000,0.995939
20,Cách nấu thịt đông kiểu truyền thống,Món Tết,2.968 kcal,70 phút,0.294890,0.239606,0.000000,0.995239


## 6. Comparison of All Methods

In [24]:
def compare_all_methods(title, df, tfidf_sim, jaccard_sim, metadata_sim, hybrid_sim, top_n=5):
    """
    Compare recommendations from all methods for a given recipe
    """
    matches = df[df['title'].str.contains(title, case=False, na=False)]
    if len(matches) == 0:
        print(f"No recipe found with title containing: {title}")
        return None
    
    recipe_idx = matches.index[0]
    
    print(f"🍽️ COMPARING RECOMMENDATIONS FOR: {df.loc[recipe_idx, 'title']}")
    
    print(f"\n📌 Recipe Details:")
    print(f"   Type: {df.loc[recipe_idx, 'type_of_food']}")
    print(f"   Calories: {df.loc[recipe_idx, 'calories']}")
    print(f"   Cook time: {df.loc[recipe_idx, 'cook_time']}")
    
    # TF-IDF Recommendations
    print("📊 METHOD 1: TF-IDF Based")
    tfidf_recs = get_tfidf_recommendations(recipe_idx, tfidf_sim, df, top_n)
    display(tfidf_recs)
    
    # Jaccard Recommendations
    print("📊 METHOD 2: Ingredient-Based (Jaccard)")
    jaccard_recs = get_ingredient_recommendations(recipe_idx, jaccard_sim, df, top_n)
    display(jaccard_recs)
    
    # Hybrid Recommendations
    print("📊 METHOD 3: Hybrid (TF-IDF: 0.4 + Jaccard: 0.4 + Metadata: 0.2)")
    hybrid_recs = get_hybrid_recommendations(recipe_idx, hybrid_sim, tfidf_sim, jaccard_sim, metadata_sim, df, top_n)
    display(hybrid_recs)
    
    return {
        'tfidf': tfidf_recs,
        'jaccard': jaccard_recs,
        'hybrid': hybrid_recs
    }

In [25]:
# Compare all methods for a sample recipe
results = compare_all_methods(
    "Canh măng", data, 
    tfidf_similarity, 
    jaccard_similarity_matrix, 
    metadata_similarity_matrix,
    hybrid_similarity_matrix,
    top_n=5
)

🍽️ COMPARING RECOMMENDATIONS FOR: Canh măng ngày Tết cổ truyền Hà Nội

📌 Recipe Details:
   Type: Món Tết
   Calories: 4.930 kcal
   Cook time: 100 phút
📊 METHOD 1: TF-IDF Based


,title,type_of_food,calories,cook_time,similarity_score
65,Canh móng giò hầm măng khô kiểu Bắc,Món Tết,2.390 kcal,90 phút,0.581095
155,Cách làm canh măng mực - đặc sản truyền thống ...,Món ngon hàng ngày,1.190 kcal,120 phút,0.484022
28,Cách làm thịt kho măng khô- món ngon Tết miền ...,Món Tết,3.043 kcal,75 phút,0.442955
330,Cách làm miến vịt măng khô đơn giản tại nhà,Món ngon hàng ngày,3.520 kcal,90 phút,0.442795
3340,"2 cách làm gà kho măng thơm ngon, bắt vị cho b...",Món Kho,NaN,30 phút + 40 phút + 15 phút + 15 phút,0.419960


📊 METHOD 2: Ingredient-Based (Jaccard)


,title,type_of_food,calories,cook_time,jaccard_score
65,Canh móng giò hầm măng khô kiểu Bắc,Món Tết,2.390 kcal,90 phút,0.090909
0,Cách muối dưa hành truyền thống,Món Tết,459 kcal,45 phút,0.000000
1,Su hào xào mực - món cổ Tết Bát Tràng,Món Tết,1.162 kcal,50 phút,0.000000
3,Giả hạnh nhân - món ngon Tết xưa Hà Nội,Món Tết,1.112 kcal,60 phút,0.000000
4,Chả bì ớt xiêm xanh,Món Tết,2.512 kcal,60 phút,0.000000


📊 METHOD 3: Hybrid (TF-IDF: 0.4 + Jaccard: 0.4 + Metadata: 0.2)


,title,type_of_food,calories,cook_time,hybrid_score,tfidf_score,jaccard_score,metadata_score
65,Canh móng giò hầm măng khô kiểu Bắc,Món Tết,2.390 kcal,90 phút,0.467437,0.581095,0.090909,0.993178
28,Cách làm thịt kho măng khô- món ngon Tết miền ...,Món Tết,3.043 kcal,75 phút,0.376339,0.442955,0.000000,0.995786
13,Cách làm canh bóng thả Hà Nội nấu theo lối xưa,Món Tết,1.157 kcal,85 phút,0.305155,0.270534,0.000000,0.984709
16,Cách làm giò xào truyền thống kiểu Bắc,Món Tết,3.082 kcal,75 phút,0.297050,0.244655,0.000000,0.995939
20,Cách nấu thịt đông kiểu truyền thống,Món Tết,2.968 kcal,70 phút,0.294890,0.239606,0.000000,0.995239


In [26]:
# Test với một món ăn khác
results2 = compare_all_methods(
    "Nem", data, 
    tfidf_similarity, 
    jaccard_similarity_matrix, 
    metadata_similarity_matrix,
    hybrid_similarity_matrix,
    top_n=5
)

🍽️ COMPARING RECOMMENDATIONS FOR: Cách làm nem rán kiểu truyền thống miền Bắc

📌 Recipe Details:
   Type: Món Tết
   Calories: 2.517 kcal
   Cook time: 65 phút
📊 METHOD 1: TF-IDF Based


,title,type_of_food,calories,cook_time,similarity_score
54,Công thức và Cách làm nem rán Hà Nội chuẩn nhất,Món Tết,2.615 kcal,75 phút,0.718620
709,"Cách làm nem cá rô giòn, ngon đổi vị ngày hè",Thực đơn cho ngày nắng nóng,2.227 kcal,70 phút,0.656962
188,Cách làm nem chim bồ câu đúng chuẩn,Món ngon hàng ngày,2.175 kcal,80 phút,0.645772
36,Cách làm nem hải sản giòn rụm cho ngày Tết,Món Tết,1.938 kcal,80 phút,0.614322
440,"Cùng làm món Nem ốc - Thơm ngon, Hấp dẫn và Độ...",Món ngon hàng ngày,1.508 kcal,90 phút,0.601300


📊 METHOD 2: Ingredient-Based (Jaccard)


,title,type_of_food,calories,cook_time,jaccard_score
54,Công thức và Cách làm nem rán Hà Nội chuẩn nhất,Món Tết,2.615 kcal,75 phút,0.153846
36,Cách làm nem hải sản giòn rụm cho ngày Tết,Món Tết,1.938 kcal,80 phút,0.142857
21,Cách làm miến xào thập cẩm ngon mà dễ làm,Món Tết,1.356 kcal,30 phút,0.136364
145,Bí quyết làm miến xào hải sản không bị dính,Món ngon hàng ngày,1.366 kcal,30 phút,0.130435
213,Thịt gà nấu đông - món ngon miền Bắc,Món ngon hàng ngày,1.439 kcal,90 phút,0.117647


📊 METHOD 3: Hybrid (TF-IDF: 0.4 + Jaccard: 0.4 + Metadata: 0.2)


,title,type_of_food,calories,cook_time,hybrid_score,tfidf_score,jaccard_score,metadata_score
54,Công thức và Cách làm nem rán Hà Nội chuẩn nhất,Món Tết,2.615 kcal,75 phút,0.548966,0.718620,0.153846,0.999896
36,Cách làm nem hải sản giòn rụm cho ngày Tết,Món Tết,1.938 kcal,80 phút,0.502754,0.614322,0.142857,0.999414
21,Cách làm miến xào thập cẩm ngon mà dễ làm,Món Tết,1.356 kcal,30 phút,0.348375,0.235900,0.136364,0.997349
26,Cách làm hạnh nhân xào - vị Tết Hà Nội xưa,Món Tết,1.323 kcal,35 phút,0.318543,0.202333,0.095238,0.997573
11,Cách làm miến xào lòng mề gà,Món Tết,1.405 kcal,45 phút,0.313058,0.192607,0.090909,0.998258


## 7. Complete Recommendation Class

In [27]:
class FoodRecommender:
    """
    Content-Based Food Recommendation System
    
    Methods:
    1. TF-IDF Based: Uses title + description + steps
    2. Ingredient Based: Uses Jaccard similarity on ingredients
    3. Hybrid: Weighted combination of TF-IDF (0.4) + Jaccard (0.4) + Metadata (0.2)
    """
    
    def __init__(self, df):
        self.df = df.copy()
        self._preprocess()
        self._build_similarity_matrices()
    
    def _preprocess(self):
        """Preprocess the data"""
        # Parse lists
        self.df['ingredients_list'] = self.df['ingredients'].apply(parse_list_string)
        self.df['step_list'] = self.df['step'].apply(parse_list_string)
        
        # Clean text
        self.df['title_clean'] = self.df['title'].apply(clean_text)
        self.df['description_clean'] = self.df['description'].fillna('').apply(clean_text)
        self.df['step_clean'] = self.df['step_list'].apply(lambda x: ' '.join([clean_text(s) for s in x]))
        self.df['combined_text'] = self.df['title_clean'] + ' ' + self.df['description_clean'] + ' ' + self.df['step_clean']
        
        # Parse numeric
        self.df['cook_time_minutes'] = self.df['cook_time'].apply(parse_cook_time)
        self.df['calories_numeric'] = self.df['calories'].apply(parse_calories)
        
        # Ingredients set
        self.df['ingredients_clean'] = self.df['ingredients_list'].apply(
            lambda x: set([clean_text(ing) for ing in x if ing])
        )
        
        print("✅ Data preprocessing completed!")
    
    def _build_similarity_matrices(self):
        """Build all similarity matrices"""
        # TF-IDF
        print("Building TF-IDF similarity matrix...")
        tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=1, max_df=0.95)
        tfidf_matrix = tfidf_vectorizer.fit_transform(self.df['combined_text'])
        self.tfidf_sim = cosine_similarity(tfidf_matrix)
        
        # Jaccard
        print("Building Jaccard similarity matrix...")
        self.jaccard_sim = compute_jaccard_similarity_matrix(self.df['ingredients_clean'].tolist())
        
        # Metadata
        print("Building Metadata similarity matrix...")
        self.metadata_sim, _ = compute_metadata_similarity_matrix(self.df)
        
        # Hybrid
        print("Building Hybrid similarity matrix...")
        self.hybrid_sim = compute_hybrid_similarity(
            self.tfidf_sim, self.jaccard_sim, self.metadata_sim,
            w_tfidf=0.4, w_jaccard=0.4, w_metadata=0.2
        )
        
        print("✅ All similarity matrices built!")
    
    def recommend(self, title, method='hybrid', top_n=5):
        """
        Get recommendations for a recipe
        
        Parameters:
        - title: Recipe title (partial match)
        - method: 'tfidf', 'jaccard', or 'hybrid'
        - top_n: Number of recommendations
        
        Returns:
        - DataFrame with recommendations
        """
        matches = self.df[self.df['title'].str.contains(title, case=False, na=False)]
        if len(matches) == 0:
            print(f"No recipe found with title containing: {title}")
            return None
        
        recipe_idx = matches.index[0]
        
        print(f"\n🍽️ Input Recipe: {self.df.loc[recipe_idx, 'title']}")
        
        if method == 'tfidf':
            return get_tfidf_recommendations(recipe_idx, self.tfidf_sim, self.df, top_n)
        elif method == 'jaccard':
            return get_ingredient_recommendations(recipe_idx, self.jaccard_sim, self.df, top_n)
        else:  # hybrid
            return get_hybrid_recommendations(
                recipe_idx, self.hybrid_sim, self.tfidf_sim, 
                self.jaccard_sim, self.metadata_sim, self.df, top_n
            )
    
    def recommend_by_index(self, idx, method='hybrid', top_n=5):
        """
        Get recommendations by recipe index
        """
        if idx < 0 or idx >= len(self.df):
            print(f"Invalid index: {idx}")
            return None
        
        print(f"\n🍽️ Input Recipe: {self.df.iloc[idx]['title']}")
        
        if method == 'tfidf':
            return get_tfidf_recommendations(idx, self.tfidf_sim, self.df, top_n)
        elif method == 'jaccard':
            return get_ingredient_recommendations(idx, self.jaccard_sim, self.df, top_n)
        else:  # hybrid
            return get_hybrid_recommendations(
                idx, self.hybrid_sim, self.tfidf_sim,
                self.jaccard_sim, self.metadata_sim, self.df, top_n
            )

In [28]:
# Create recommender instance
print("Initializing Food Recommender...")
recommender = FoodRecommender(df)

Initializing Food Recommender...
✅ Data preprocessing completed!
Building TF-IDF similarity matrix...
✅ Data preprocessing completed!
Building TF-IDF similarity matrix...
Building Jaccard similarity matrix...
Building Jaccard similarity matrix...
Building Metadata similarity matrix...
Metadata features shape: (3524, 28)
Building Hybrid similarity matrix...
Weights: TF-IDF=0.40, Jaccard=0.40, Metadata=0.20
Building Metadata similarity matrix...
Metadata features shape: (3524, 28)
Building Hybrid similarity matrix...
Weights: TF-IDF=0.40, Jaccard=0.40, Metadata=0.20
✅ All similarity matrices built!
✅ All similarity matrices built!


In [29]:
# Test the recommender
print("TESTING FOOD RECOMMENDER")

# TF-IDF method
print("\n📊 TF-IDF Method:")
display(recommender.recommend("Thịt kho", method='tfidf', top_n=5))

# Jaccard method
print("\n📊 Jaccard Method:")
display(recommender.recommend("Thịt kho", method='jaccard', top_n=5))

# Hybrid method
print("\n📊 Hybrid Method:")
display(recommender.recommend("Thịt kho", method='hybrid', top_n=5))

TESTING FOOD RECOMMENDER

📊 TF-IDF Method:

🍽️ Input Recipe: Cách làm thịt kho hột vịt truyền thống miền Tây


,title,type_of_food,calories,cook_time,similarity_score
46,Thịt kho tàu - món ăn đặc trưng Tết Nam Bộ,Món Tết,3.998 kcal,30 phút,0.556325
57,Thịt kho tàu kiểu Bắc – món ngon Tết đến,Món Tết,2.270 kcal,60 phút,0.418019
2718,Thịt kho nước dừa,api,NaN,60 phút,0.401745
3387,Cách ướp thịt vịt nướng tại nhà thơm ngon chuẩ...,Món Nướng,NaN,30 phút + 2 giờ,0.389416
28,Cách làm thịt kho măng khô- món ngon Tết miền ...,Món Tết,3.043 kcal,75 phút,0.383531



📊 Jaccard Method:

🍽️ Input Recipe: Cách làm thịt kho hột vịt truyền thống miền Tây


,title,type_of_food,calories,cook_time,jaccard_score
392,Cách làm thịt kho măng đơn giản mà siêu ngon t...,Món ngon hàng ngày,2.034 kcal,50 phút,0.100000
144,Cách làm thịt ba chỉ kho trứng cút đậm đà,Món ngon hàng ngày,1.908 kcal,50 phút,0.083333
2734,Xôi viên ngọc,api,NaN,30 phút,0.083333
1296,Bánh ít trần,api,NaN,30 phút,0.062500
3232,Mực nướng tương hột,api,NaN,30 phút,0.062500



📊 Hybrid Method:

🍽️ Input Recipe: Cách làm thịt kho hột vịt truyền thống miền Tây


,title,type_of_food,calories,cook_time,hybrid_score,tfidf_score,jaccard_score,metadata_score
46,Thịt kho tàu - món ăn đặc trưng Tết Nam Bộ,Món Tết,3.998 kcal,30 phút,0.421205,0.556325,0.0,0.993373
57,Thịt kho tàu kiểu Bắc – món ngon Tết đến,Món Tết,2.270 kcal,60 phút,0.366881,0.418019,0.0,0.998365
28,Cách làm thịt kho măng khô- món ngon Tết miền ...,Món Tết,3.043 kcal,75 phút,0.353259,0.383531,0.0,0.999231
50,Cá trắm đen kho riềng - món ngon Tết xưa Hà Nội,Món Tết,1.792 kcal,240 phút,0.327455,0.327518,0.0,0.982241
20,Cách nấu thịt đông kiểu truyền thống,Món Tết,2.968 kcal,70 phút,0.319817,0.300025,0.0,0.999034


## 8. Summary

### Các phương pháp đã implement:

1. **TF-IDF Based Recommendation**
   - Sử dụng text vectorization với TF-IDF
   - Features: title + description + cooking steps
   - Similarity: Cosine similarity

2. **Ingredient-Based Recommendation**
   - So sánh danh sách nguyên liệu
   - Similarity: Jaccard similarity
   - $J(A,B) = \frac{|A \cap B|}{|A \cup B|}$

3. **Hybrid Approach**
   - Weighted combination của các phương pháp
   - Weights: TF-IDF (0.4) + Jaccard (0.4) + Metadata (0.2)
   - Metadata bao gồm: calories, cook_time, type_of_food

### Next Steps:
- Evaluation metrics (Precision, Recall, NDCG)
- A/B Testing
- User feedback integration
- Deep learning approaches (Embeddings, Neural Collaborative Filtering)

## 9. Export Functions cho Team Evaluation

Các hàm để team Evaluation có thể sử dụng để đánh giá hệ thống

In [30]:
def export_recommendations_for_evaluation(recommender, sample_indices, methods=['tfidf', 'jaccard', 'hybrid'], top_n=10):
    """
    Export recommendations cho team Evaluation
    
    Parameters:
    - recommender: FoodRecommender instance
    - sample_indices: List các index của món ăn cần test
    - methods: List các phương pháp cần đánh giá
    - top_n: Số lượng recommendations
    
    Returns:
    - Dictionary chứa recommendations cho từng method
    """
    results = {method: [] for method in methods}
    
    for idx in sample_indices:
        recipe_title = recommender.df.iloc[idx]['title']
        
        for method in methods:
            recs = recommender.recommend_by_index(idx, method=method, top_n=top_n)
            if recs is not None:
                rec_data = {
                    'query_idx': idx,
                    'query_title': recipe_title,
                    'recommendations': recs['title'].tolist(),
                    'scores': recs[f'{method}_score' if method != 'hybrid' else 'hybrid_score'].tolist() if method != 'hybrid' else recs['hybrid_score'].tolist()
                }
                results[method].append(rec_data)
    
    return results

def get_all_similarity_scores(recommender, query_idx, top_n=None):
    """
    Lấy tất cả similarity scores cho một món ăn
    Dùng để team Evaluation tính các metrics
    
    Parameters:
    - recommender: FoodRecommender instance
    - query_idx: Index của món ăn query
    - top_n: Số lượng top results (None = tất cả)
    
    Returns:
    - DataFrame với tất cả scores
    """
    n_recipes = len(recommender.df)
    
    results = pd.DataFrame({
        'idx': range(n_recipes),
        'title': recommender.df['title'].values,
        'type_of_food': recommender.df['type_of_food'].values,
        'tfidf_score': recommender.tfidf_sim[query_idx],
        'jaccard_score': recommender.jaccard_sim[query_idx],
        'metadata_score': recommender.metadata_sim[query_idx],
        'hybrid_score': recommender.hybrid_sim[query_idx]
    })
    
    # Loại bỏ chính nó
    results = results[results['idx'] != query_idx]
    
    # Sort by hybrid score
    results = results.sort_values('hybrid_score', ascending=False)
    
    if top_n:
        results = results.head(top_n)
    
    return results.reset_index(drop=True)

def export_to_csv(recommender, output_dir='../evaluation_data'):
    """
    Export tất cả similarity matrices và data ra CSV cho team Evaluation
    
    Parameters:
    - recommender: FoodRecommender instance
    - output_dir: Thư mục lưu file
    """
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    # Export recipe data
    recipe_data = recommender.df[['title', 'type_of_food', 'calories', 'cook_time', 'source']].copy()
    recipe_data.to_csv(f'{output_dir}/recipes_info.csv', index=True, encoding='utf-8-sig')
    print(f"✅ Saved: {output_dir}/recipes_info.csv")
    
    # Export similarity matrices
    np.save(f'{output_dir}/tfidf_similarity.npy', recommender.tfidf_sim)
    print(f"✅ Saved: {output_dir}/tfidf_similarity.npy")
    
    np.save(f'{output_dir}/jaccard_similarity.npy', recommender.jaccard_sim)
    print(f"✅ Saved: {output_dir}/jaccard_similarity.npy")
    
    np.save(f'{output_dir}/metadata_similarity.npy', recommender.metadata_sim)
    print(f"✅ Saved: {output_dir}/metadata_similarity.npy")
    
    np.save(f'{output_dir}/hybrid_similarity.npy', recommender.hybrid_sim)
    print(f"✅ Saved: {output_dir}/hybrid_similarity.npy")
    
    print(f"\n✅ Đã export tất cả data cho team Evaluation vào thư mục: {output_dir}")

print("✅ Export functions đã sẵn sàng!")

✅ Export functions đã sẵn sàng!


In [31]:
# Test export functions
print("📊 Test get_all_similarity_scores:")
scores_df = get_all_similarity_scores(recommender, query_idx=0, top_n=10)
display(scores_df)

📊 Test get_all_similarity_scores:


,idx,title,type_of_food,tfidf_score,jaccard_score,metadata_score,hybrid_score
0,52,"Cách muối hành trắng giòn, để được lâu",Món Tết,0.570477,0.0,0.998228,0.427836
1,34,Cách làm dưa món giòn ngon đón Tết,Món Tết,0.295631,0.0,0.999347,0.318122
2,53,Cách làm dưa góp giòn ngon cho ngày Tết,Món Tết,0.287602,0.0,0.999773,0.314996
3,30,Cách làm nộm tai heo dưa chuột giải ngán ngày Tết,Món Tết,0.282646,0.0,0.999870,0.313032
4,43,Nộm gà hoa chuối giòn ngon đổi vị ngày Tết,Món Tết,0.279169,0.0,0.999233,0.311514
5,20,Cách nấu thịt đông kiểu truyền thống,Món Tết,0.243763,0.0,0.992451,0.295995
6,64,Cách làm mứt cà rốt không cần nước vôi trong,Món Tết,0.241383,0.0,0.995738,0.295701
7,23,Cách làm gà bóp hành răm kiểu miền Trung,Món Tết,0.238090,0.0,0.999941,0.295224
8,33,Cách làm tré Huế - đặc sản cố đô vào dịp Tết,Món Tết,0.235693,0.0,0.997707,0.293819
9,42,Thịt ngâm mắm - món ngon miền Trung,Món Tết,0.230432,0.0,0.993687,0.290910


In [33]:
# Export data cho team Evaluation (uncomment để chạy)
# export_to_csv(recommender, output_dir='../evaluation_data')

#"Để export data cho team Evaluation, uncomment dòng trên và chạy lại cell này"

## 10. Hướng dẫn sử dụng cho các phần sau

### Cho phần Evaluation (Phần C):
```python
# Load similarity matrices đã export
import numpy as np
import pandas as pd

tfidf_sim = np.load('../evaluation_data/tfidf_similarity.npy')
jaccard_sim = np.load('../evaluation_data/jaccard_similarity.npy')
hybrid_sim = np.load('../evaluation_data/hybrid_similarity.npy')
recipes_info = pd.read_csv('../evaluation_data/recipes_info.csv')

# Tính Precision@K, Recall@K, NDCG@K, MRR từ các similarity matrices này
```

### Cho phần LLM-Based (Phần B):
```python
# Có thể dùng hybrid_similarity làm baseline để so sánh với LLM approach
# Hoặc dùng kết quả Content-Based làm input cho LLM Reranking
```